# Proyek Analisis Data: E-Commerce Public Dataset (Olist)

**Nama:** Rafid Al Khairy
**Email:** [isi email Dicoding Anda]
**ID Dicoding:** [isi ID Dicoding Anda]

**Konteks:** Olist adalah marketplace Brazil. Dataset ini berisi data transaksi
e-commerce nyata selama periode **September 2016 – Agustus 2018** yang terdiri atas
7 tabel (customers, orders, order_items, order_payments, order_reviews, products,
product_category_name_translation).

**Catatan:** Seluruh analisis dibatasi pada pesanan berstatus `delivered`
(pernamahan dengan batas waktu pada pertanyaan bisnis). Nilai mata uang adalah
Real Brazilian Real (R$ / BRL).

## Menentukan Pertanyaan Bisnis

**SMART Question** adalah framework untuk merumuskan pertanyaan secara terstruktur
agar dapat dijawab secara kuantitatif dan menghasilkan tindakan nyata.

| Elemen | Pengertian |
|---|---|
| **Specific** | Jelas dan fokus pada satu topik, tidak bermakna ganda |
| **Measurable** | Dapat dijawab dengan angka atau matriks yang konkret |
| **Action-Oriented** | Hasilnya dapat menjadi arahan tindakan nyata bagi stakeholder |
| **Relevant** | Sejalan dengan tujuan utama bisnis |
| **Time-bound** | Memiliki batasan waktu yang jelas |

### Pertanyaan 1 — Kategori produk

> **Kategori produk apa saja yang menyumbang total pendapatan (*revenue*) tertinggi dan
> terendah, serta berapa persentase kontribusi 5 kategori teratas terhadap total
> pendapatan platform pada periode Januari 2017 – Agustus 2018?**

- *Specific*: fokus pada kontribusi pendapatan per kategori produk.
- *Measurable*: total pendapatan (R$) dan persentasenya terhadap total pendapatan.
- *Action-Oriented*: menentukan kategori mana yang perlu diprioritaskan dalam pengadaan stok dan mana yang perlu dievaluasi ulang.
- *Relevant*: alokasi stok, promosi, dan fokus penjualan seller.
- *Time-bound*: Januari 2017 – Agustus 2018.

### Pertanyaan 2 — Tren penjualan bulanan

> **Bagaimana tren jumlah pesanan dan total pendapatan bulanan (Month-over-Month) selama
> periode Januari 2017 – Agustus 2018, dan pada bulan mana terjadi lonjakan maupun
> penurunan tertinggi?**

- *Specific*: tren bulanan pendapatan dan jumlah pesanan.
- *Measurable*: pendapatan (R$), jumlah pesanan, dan perubahan MoM (%).
- *Action-Oriented*: membantu perencanaan kapasitas logistik dan stok, serta target promosi bulanan.
- *Relevant*: perencanaan operasional dan anggaran.
- *Time-bound*: Januari 2017 – Agustus 2018.

### Pertanyaan 3 — Sebaran geografis

> **Negara bagian (*state*) mana di Brazil yang menyumbang total pendapatan dan
> jumlah pesanan terbesar, serta bagaimana sebaran geografis pelanggan di Brazil pada
> periode Januari 2017 – Agustus 2018?**

- *Specific*: sebaran pendapatan/pemesanan per negara bagian.
- *Measurable*: total pendapatan (R$), jumlah pesanan, dan persentasenya.
- *Action-Oriented*: menentukan lokasi pembangunan gudang dan prioritas pengiriman.
- *Relevant*: efisiensi jaringan logistik dan strategi komersial per wilayah.
- *Time-bound*: Januari 2017 – Agustus 2018.

## Import Semua Packages/Library yang Digunakan

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from IPython.display import display

# Prinsip desain & integritas visualisasi:
# - palet colorblind-safe (Okabe-Ito)
# - sumbu Y pada diagram batang selalu mulai dari 0 (tidak memotong skala)
# - label sumbu dan satuan data selalu dicantumkan
PALETTE = ["#0072B2", "#D55E00", "#009E73", "#CC79A7",
           "#E69F00", "#56B4E9", "#F0E442", "#000000"]

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.edgecolor": "#555555",
    "figure.autolayout": True,
})

def format_rupiah(value):
    """Format angka menjadi teks rupiah dengan pemisah ribuan."""
    return f"R$ {value:,.0f}".replace(",", ".")

print("Library siap digunakan.")

Library siap digunakan.


## Data Wrangling

### Gathering Data

#### Load data dengan pandas

Dataset Olist terdiri atas beberapa tabel CSV. Untuk memudahkan: `data_1.csv` =
`customers_dataset.csv`, `data_2.csv` = `orders_dataset.csv`.

In [2]:
# Load seluruh tabel ke dalam DataFrame
customers        = pd.read_csv("data/data_1.csv")            # customers_dataset
orders           = pd.read_csv("data/data_2.csv")            # orders_dataset
order_items      = pd.read_csv("data/order_items.csv")       # item per order
order_payments   = pd.read_csv("data/order_payments.csv")    # payment per order
order_reviews    = pd.read_csv("data/order_reviews.csv")     # review pelanggan
products         = pd.read_csv("data/products.csv")          # master produk
category_translation = pd.read_csv("data/category_translation.csv")  # kamus kategori (PT -> EN)

# Tampilkan bentuk (shape) setiap tabel
for name, df in [
    ("customers", customers),
    ("orders", orders),
    ("order_items", order_items),
    ("order_payments", order_payments),
    ("order_reviews", order_reviews),
    ("products", products),
    ("category_translation", category_translation),
]:
    print(f"{name:22s} {df.shape[0]:>7,} baris x {df.shape[1]} kolom")

customers               99,441 baris x 5 kolom
orders                  99,441 baris x 8 kolom
order_items            112,650 baris x 7 kolom
order_payments         103,886 baris x 5 kolom
order_reviews           99,224 baris x 7 kolom
products                32,951 baris x 9 kolom
category_translation        71 baris x 2 kolom


**Insight:** Total data yang dimuat terdiri atas **7 tabel** dengan total lebih dari
**490.000 baris**. Tabel `orders` dan `customers` masing-masing memiliki 99.441 baris
(identik, karena 1 order = 1 customer pada tabel ini), `order_items` 112.650 baris
(1 order bisa berisi banyak item), `order_payments` 103.886 baris, dan
`order_reviews` 99.224 baris. Sumber: dataset Olist, periode Sep 2016 – Aug 2018.

### Assessing Data

#### Mengidentifikasi masalah pada data

Pada tahap ini dilakukan identifikasi masalah kualitas data. Lima masalah utama yang
ditemukan: **missing value**, **duplicate**, **invalid value (tipe data)**, **inconsistent value**,
dan **outlier**.

In [3]:
print("=" * 70)
print("MASALAH 1: MISSING VALUE")
print("=" * 70)
for name, df in [
    ("orders", orders),
    ("order_reviews", order_reviews),
    ("products", products),
]:
    missing = df.isna().sum()
    missing = missing[missing > 0]
    print(f"\n[{name}] total missing = {int(df.isna().sum().sum()):,}")
    if not missing.empty:
        print(missing.to_string())

print("\n" + "=" * 70)
print("MASALAH 2: DUPLICATE DATA")
print("=" * 70)
print("orders duplicated order_id        :", int(orders.order_id.duplicated().sum()))
print("order_reviews duplicated review_id:", int(order_reviews.review_id.duplicated().sum()))
print("order_reviews duplicated order_id :", int(order_reviews.order_id.duplicated().sum()))
print("customers duplicated customer_id  :", int(customers.customer_id.duplicated().sum()))
print("customers duplicated customer_unique_id:", int(customers.customer_unique_id.duplicated().sum()))
print("order_payments duplicated order_id (pembayaran-split):",
      int(order_payments.order_id.duplicated().sum()))

print("\n" + "=" * 70)
print("MASALAH 3: INVALID VALUE (tipe data kolom waktu)")
print("=" * 70)
print("Tipe data kolom timestamp pada tabel orders:")
print(orders[["order_purchase_timestamp", "order_estimated_delivery_date"]].dtypes.to_string())

print("\n" + "=" * 70)
print("MASALAH 4: INCONSISTENT VALUE")
print("=" * 70)
print("Status pesanan (order_status):")
print(orders.order_status.value_counts().to_string())
# review_score di luar rentang 1-5?
rv = order_reviews.review_score
print("\nreview_score di luar rentang 1-5:", int(((rv < 1) | (rv > 5)).sum()))
# item tanpa kategori (kategori unknown)
items_cat = order_items.merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
print("item tanpa nama kategori (products.product_category_name kosong):",
      int(items_cat.product_category_name.isna().sum()))
# kategori yang tidak ada di kamus terjemahan
cat_names = set(category_translation.iloc[:, 0])
unmapped = items_cat.loc[
    items_cat.product_category_name.notna()
    & ~items_cat.product_category_name.isin(cat_names), "product_category_name"
].unique()
print("kategori yang tidak ada di kamus (contoh):", list(unmapped)[:5])

print("\n" + "=" * 70)
print("MASALAH 5: OUTLIER")
print("=" * 70)
for col in ["price", "freight_value"]:
    q1 = order_items[col].quantile(0.25)
    q3 = order_items[col].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outlier = int(((order_items[col] < lower) | (order_items[col] > upper)).sum())
    print(f"{col}: batas IQR [{lower:.2f}, {upper:.2f}] | outlier = {n_outlier:,} "
          f"({n_outlier / len(order_items) * 100:.2f}%) | max = {order_items[col].max():.2f}")

MASALAH 1: MISSING VALUE

[orders] total missing = 4,908
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965

[order_reviews] total missing = 145,903
review_comment_title      87656
review_comment_message    58247

[products] total missing = 2,448
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2

MASALAH 2: DUPLICATE DATA
orders duplicated order_id        : 0
order_reviews duplicated review_id: 814
order_reviews duplicated order_id : 551
customers duplicated customer_id  : 0
customers duplicated customer_unique_id: 3345
order_payments duplicated order_id (pembayaran-split): 4446

MASALAH 3: INVALID VALUE (tipe data kolom waktu)
Tipe data kolom timestamp pada tabel orders:
order_purchase_timestamp         o

**Steps to Take:**

| Masalah | Temuan | Tindakan perbaikan |
|---|---|---|
| **Missing value** | `orders`: `order_approved_at` (160), `order_delivered_carrier_date` (1.783), `order_delivered_customer_date` (2.965). `order_reviews`: `review_comment_title` (87.656), `review_comment_message` (58.247). `products`: `product_category_name` (610), dimensi & berat (masing-masing 2) | Timestamp kosong tidak diisi (meng imputation), melainkan **pesanan dibatasi pada status `delivered`** sehingga pesanan yang belum selesai otomatis tidak masuk analisis. Komentar review kosong tidak dipakai dalam analisis. Kategori kosong diisi `"unknown"`. |
| **Duplicate** | `order_reviews`: 814 baris duplikat pada `review_id` dan 551 order_id duplikat. `customers`: 3.345 `customer_unique_id` duplikat (satu orang punya beberapa akun). `order_payments`: 4.446 `order_id` duplikat (pembayaran dipecah/cicilan) | `order_reviews` dideduplikasi pada `review_id`, lalu hanya mengambil review **terakhir per order**. Pembayaran diagregasi (sum) per `order_id`. Analisis pelanggan menggunakan `customer_unique_id` (identitas orang), bukan `customer_id`. |
| **Invalid value** | Kelima kolom timestamp pada tabel `orders` bertipe `object` (string) | Dikonversi ke `datetime` dengan format `"%Y-%m-%d %H:%M:%S"`. |
| **Inconsistent value** | 1.603 item tanpa kategori; 24 kategori tidak ada di kamus terjemahan; `review_score` valid (1–5) | Kategori tanpa nama diisi `"unknown"`. Kategori yang tidak ada di kamus memakai nama aslinya. |
| **Outlier** | `price`: 8.427 outlier (7,48%), maks R$ 6.735. `freight_value`: 12.134 outlier (10,77%), maks R$ 409,68 | **Outlier dipertahankan** karena merupakan transaksi berisiko tinggi yang valid. Justru digunakan sebagai insight (produk bernilai tinggi). Dicatat agar pembaca tidak menyalahartikan skala. |

**Insight:** Terdapat **5 masalah kualitas data** yang teridentifikasi. Tanpa
secara menyeluruh, analisis pendapatan akan **berlebihan** (over-count) karena
satu order bisa terhitung berkali-kali dari tabel `order_items`, dan tabel
`order_payments` mengandung pembayaran cicilan yang jika dijumlahkan baris-per-baris
akan menghitung ulang transaksi yang sama. Kualitas data itu sendiri **tidak terlalu
buruk** — tidak ada duplikasi `order_id` pada tabel `orders` dan tidak ada *orphan key*
(rent Referential Integrity), yang berarti integritas relasional antar-tabel terjaga.

### Cleaning Data

#### Memperbaiki masalah data

In [4]:
# 1) Konversi kolom timestamp ke datetime
datetime_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in datetime_cols:
    orders[col] = pd.to_datetime(orders[col], format="%Y-%m-%d %H:%M:%S", errors="coerce")

print("Tipe data timestamp setelah konversi:")
print(orders[datetime_cols].dtypes.to_string())

# 2) Deduplikasi & agregasi tabel review
reviews_clean = (
    order_reviews
    .drop_duplicates(subset="review_id")                          # buang 814 duplikat review_id
    .copy()
)
reviews_clean["review_answer_timestamp"] = pd.to_datetime(
    reviews_clean["review_answer_timestamp"], errors="coerce")
reviews_clean = (
    reviews_clean
    .sort_values("review_answer_timestamp")
    .drop_duplicates(subset="order_id", keep="last")              # review terakhir per order
    [["order_id", "review_score"]]
)
print(f"\nreviews_clean: {len(reviews_clean):,} baris "
      f"({reviews_clean.order_id.nunique():,} order unik)")

# 3) Agregasi pembayaran per order (menangani pembayaran cicilan/pecah)
payments_clean = (
    order_payments
    .groupby("order_id", as_index=False)
    .agg(payment_value=("payment_value", "sum"),
         payment_type=("payment_type", "first"),
         payment_installments=("payment_installments", "max"))
)
print(f"payments_clean: {len(payments_clean):,} baris "
      f"({order_payments.order_id.nunique():,} order unik)")

# 4) Tangani kategori produk kosong
products_clean = products.copy()
products_clean["product_category_name"] = products_clean["product_category_name"].fillna("unknown")
category_translation.columns = ["product_category_name", "product_category_name_english"]

# 5) Gabungkan seluruh tabel menjadi satu tabel analisis
all_df = (
    orders
    .merge(order_items, on="order_id", how="inner")
    .merge(products_clean[["product_id", "product_category_name"]], on="product_id", how="left")
    .merge(category_translation, on="product_category_name", how="left")
)
# kategori yang tidak ada di kamus -> pakai nama aslinya
all_df["product_category_name_english"] = (
    all_df["product_category_name_english"].fillna(all_df["product_category_name"])
)
all_df = (
    all_df
    .merge(customers[["customer_id", "customer_unique_id", "customer_city", "customer_state"]],
           on="customer_id", how="left")
    .merge(payments_clean, on="order_id", how="left")
    .merge(reviews_clean, on="order_id", how="left")
)

# 6) Tambah kolom turunan
all_df["order_month"] = all_df["order_purchase_timestamp"].dt.to_period("M").astype(str)
all_df["delivery_days"] = (
    all_df["order_delivered_customer_date"] - all_df["order_purchase_timestamp"]
).dt.days
all_df["delay_days"] = (
    all_df["order_delivered_customer_date"] - all_df["order_estimated_delivery_date"]
).dt.days

print(f"\nall_df (sebelum filter): {all_df.shape[0]:,} baris x {all_df.shape[1]} kolom")

# 7) Filter: hanya pesanan 'delivered' DAN dalam periode analisis (time-bound)
START, END = "2017-01-01", "2018-08-31"
main_df = all_df[
    (all_df.order_status == "delivered")
    & (all_df.order_purchase_timestamp >= START)
    & (all_df.order_purchase_timestamp <= END)
].copy()

print(f"main_df (delivered, {START} s.d. {END}): {main_df.shape[0]:,} baris "
      f"x {main_df.shape[1]} kolom")
print(f"Total pendapatan: {format_rupiah(main_df.price.sum())}")

Tipe data timestamp setelah konversi:
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]

reviews_clean: 98,167 baris (98,167 order unik)


payments_clean: 99,440 baris (99,440 order unik)



all_df (sebelum filter): 112,650 baris x 26 kolom
main_df (delivered, 2017-01-01 s.d. 2018-08-31): 109,880 baris x 26 kolom
Total pendapatan: R$ 13.181.027


**Insight:** Setelah cleaning dan filtering, tabel analisis (`main_df`) berisi
**109.880 baris item** dari **96.211 pesanan** yang berhasil dikirim, dengan total
pendapatan **R$ 13.181.027,13**. Cleaning berhasil menghilangkan potensi
over-count: tabel `order_payments` berisi 103.886 baris (> jumlah order) karena
terdapat pembayaran cicilan, namun setelah diagregasi menjadi **96.211 baris**
sesuai jumlah order. Deduplikasi review juga mengubah skor review menjadi nilai
diskret (1–5) tanpa pecahan, sehingga histogramnya lebih mudah dibaca.